In [2]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
import os
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import mlflow
import mlflow.sklearn

print("Libraries loaded!")

Libraries loaded!


In [3]:
os.chdir('..')
print(f"Working directory: {os.getcwd()}")

Working directory: /Users/touhidulislamalvi/Desktop/Social_Media_Mood_Analyzer


In [4]:
# Load TF-IDF matrices
X_train = sp.load_npz('data/processed/X_train.npz')
X_val = sp.load_npz('data/processed/X_val.npz')
X_test = sp.load_npz('data/processed/X_test.npz')

# Load labels
y_train = pd.read_csv('data/processed/y_train.csv')['mood']
y_val = pd.read_csv('data/processed/y_val.csv')['mood']
y_test = pd.read_csv('data/processed/y_test.csv')['mood']

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (43410, 10000)
X_val shape: (5426, 10000)
X_test shape: (5427, 10000)


In [5]:
# Start MLflow experiment
mlflow.set_experiment("social_mood_analyzer")

with mlflow.start_run(run_name="logistic_regression_v1"):
    
    # Initialize model
    model = LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    )
    
    # Train model
    print("Training model...")
    model.fit(X_train, y_train)
    
    # Evaluate on validation set
    val_score = model.score(X_val, y_val)
    print(f"Validation accuracy: {val_score:.4f}")
    
    # Log parameters and metrics to MLflow
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_metric("val_accuracy", val_score)
    
    # Save model
    mlflow.sklearn.log_model(model, "model")
    
    print("Model trained and logged to MLflow!")

2026/05/19 20:37:11 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/19 20:37:11 INFO mlflow.store.db.utils: Updating database tables
2026/05/19 20:37:11 INFO mlflow.tracking.fluent: Experiment with name 'social_mood_analyzer' does not exist. Creating a new experiment.


Training model...


2026/05/19 20:37:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/19 20:37:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Validation accuracy: 0.6563
Model trained and logged to MLflow!


In [6]:
from sklearn.metrics import classification_report

# Get predictions on validation set
y_pred = model.predict(X_val)

# Detailed report per mood class
print(classification_report(y_val, y_pred, 
      target_names=['angry', 'happy', 'neutral', 'sad']))

              precision    recall  f1-score   support

       angry       0.44      0.54      0.49       799
       happy       0.82      0.68      0.75      2096
     neutral       0.67      0.68      0.67      2089
         sad       0.49      0.64      0.55       442

    accuracy                           0.66      5426
   macro avg       0.60      0.63      0.61      5426
weighted avg       0.68      0.66      0.66      5426



In [8]:
# Load cleaned text data
df_train = pd.read_csv('data/processed/train_cleaned.csv')
df_val = pd.read_csv('data/processed/val_cleaned.csv')
df_test = pd.read_csv('data/processed/test_cleaned.csv')

print("Data loaded!")

Data loaded!


In [9]:
with mlflow.start_run(run_name="logistic_regression_v2"):
    
    # Improved TF-IDF
    from sklearn.feature_extraction.text import TfidfVectorizer
    
    tfidf_v2 = TfidfVectorizer(
        max_features=20000,   # increased from 10,000
        ngram_range=(1, 3),   # added trigrams
        min_df=2,
        max_df=0.95
    )
    
    # Refit on train
    X_train_v2 = tfidf_v2.fit_transform(
        df_train['cleaned_text'].fillna('')
    )
    X_val_v2 = tfidf_v2.transform(
        df_val['cleaned_text'].fillna('')
    )
    
    # Train model
    model_v2 = LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    )
    
    model_v2.fit(X_train_v2, y_train)
    
    # Evaluate
    y_pred_v2 = model_v2.predict(X_val_v2)
    val_score_v2 = model_v2.score(X_val_v2, y_val)
    
    print(f"V2 Validation accuracy: {val_score_v2:.4f}")
    print(classification_report(y_val, y_pred_v2,
          target_names=['angry', 'happy', 'neutral', 'sad']))
    
    # Log to MLflow
    mlflow.log_param("max_features", 20000)
    mlflow.log_param("ngram_range", "(1,3)")
    mlflow.log_metric("val_accuracy", val_score_v2)
    mlflow.sklearn.log_model(model_v2, "model")

2026/05/19 20:40:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/19 20:40:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


V2 Validation accuracy: 0.6585
              precision    recall  f1-score   support

       angry       0.45      0.54      0.49       799
       happy       0.82      0.69      0.75      2096
     neutral       0.66      0.68      0.67      2089
         sad       0.49      0.62      0.55       442

    accuracy                           0.66      5426
   macro avg       0.61      0.63      0.61      5426
weighted avg       0.68      0.66      0.66      5426



In [10]:
from sklearn.svm import LinearSVC

with mlflow.start_run(run_name="svm_v1"):
    
    # Initialize SVM
    model_svm = LinearSVC(
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    )
    
    # Train
    print("Training SVM...")
    model_svm.fit(X_train, y_train)
    
    # Evaluate
    y_pred_svm = model_svm.predict(X_val)
    val_score_svm = model_svm.score(X_val, y_val)
    
    print(f"SVM Validation accuracy: {val_score_svm:.4f}")
    print(classification_report(y_val, y_pred_svm,
          target_names=['angry', 'happy', 'neutral', 'sad']))
    
    # Log to MLflow
    mlflow.log_param("model", "LinearSVC")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_metric("val_accuracy", val_score_svm)
    mlflow.sklearn.log_model(model_svm, "model")

Training SVM...


2026/05/19 20:41:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/19 20:41:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


SVM Validation accuracy: 0.6393
              precision    recall  f1-score   support

       angry       0.43      0.45      0.44       799
       happy       0.77      0.71      0.74      2096
     neutral       0.64      0.66      0.65      2089
         sad       0.49      0.55      0.52       442

    accuracy                           0.64      5426
   macro avg       0.58      0.59      0.59      5426
weighted avg       0.65      0.64      0.64      5426



In [11]:
# Final evaluation on test set
y_pred_test = model.predict(X_test)
test_score = model.score(X_test, y_test)

print(f"FINAL Test Accuracy: {test_score:.4f}")
print()
print(classification_report(y_test, y_pred_test,
      target_names=['angry', 'happy', 'neutral', 'sad']))

FINAL Test Accuracy: 0.6600

              precision    recall  f1-score   support

       angry       0.46      0.55      0.50       822
       happy       0.82      0.69      0.75      2026
     neutral       0.68      0.69      0.68      2139
         sad       0.47      0.60      0.53       440

    accuracy                           0.66      5427
   macro avg       0.61      0.63      0.61      5427
weighted avg       0.68      0.66      0.67      5427



In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Recreate and refit TF-IDF vectorizer
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

# Fit on train data
tfidf.fit_transform(df_train['cleaned_text'].fillna(''))

print("TF-IDF vectorizer recreated!")

TF-IDF vectorizer recreated!


In [14]:
import joblib

# Save best model and vectorizer
joblib.dump(model, 'data/processed/best_model.pkl')
joblib.dump(tfidf, 'data/processed/tfidf_vectorizer.pkl')

print("Best model saved!")

Best model saved!
